# From Noise to a Spiral: Diffusion from Scratch

> **Solutions.** Teacher copy, not for posting.

**Day 9 · Human and Machine Perception**

A diffusion model learns to undo noise. Training: take real data, add noise, and teach a network to predict the noise you added. Generation: start from pure noise and remove the predicted noise a little at a time, until data appears.

We do it on 2D points shaped like a spiral, so every step is something you can plot.

**How to use it:** run the cells top to bottom. When a cell stops with `NotImplementedError`, that is a **TODO** for you. Write the missing lines where the `# TODO` comment is, delete the `raise` line, and run the cell again.

| TODO | Where | What you write |
|---|---|---|
| 1 | 2 | Add noise to the data |
| 2 | 4 | The training loss |
| 3 | 5 | One step of the sampler |
| 4 | 7, if you have time | A faster sampler |

Training runs on a CPU in about a minute.

*The setup follows the design of Chenyang Yuan's [smalldiffusion](https://github.com/yuanchenyang/smalldiffusion) library. The code here is written from scratch in plain PyTorch.*

## Setup

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
print("Running on CPU")

---
## 1. The data: 200 points on a spiral

Each data point is just two numbers, $x_0 \in \mathbb{R}^2$. The model never sees the formula for the spiral, only these points.

In [ ]:
def spiral(n=200, start=np.pi / 2, end=5 * np.pi):
    t = np.linspace(start, end, n)
    pts = np.stack([t * np.cos(t), t * np.sin(t)], axis=1) / end   # scale to about [-1, 1]
    return torch.tensor(pts, dtype=torch.float32)

data = spiral()
def plot_points(pts, ax=None, **kw):
    ax = ax or plt.gca()
    pts = pts.detach().cpu().numpy()
    ax.scatter(pts[:, 0], pts[:, 1], s=kw.pop("s", 6), **kw)
    ax.set_aspect("equal"); ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3)

plt.figure(figsize=(4, 4)); plot_points(data); plt.title("200 real points"); plt.show()

---
## 2. Adding noise

The noise level is a number $\sigma$. A noisy version of a point is

$$x_\sigma = x_0 + \sigma\,\varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, I)$$

Small $\sigma$ barely moves the point. Large $\sigma$ buries it: at $\sigma = 10$ the spiral is gone. We use 200 noise levels, spaced evenly on a log scale from $\sigma_\text{max} = 10$ down to $\sigma_\text{min} = 0.005$.

In [ ]:
SIGMA_MAX, SIGMA_MIN, N_LEVELS = 10.0, 0.005, 200
sigmas = torch.exp(torch.linspace(math.log(SIGMA_MAX), math.log(SIGMA_MIN), N_LEVELS))

def add_noise(x0, sigma):
    # x0: points, shape (n, 2). sigma: a number, or a tensor of shape (n, 1).
    # Returns the noisy points and the noise eps that was added.
    eps = torch.randn_like(x0)
    # TODO: the noisy points x0 + sigma * eps
    x_noisy = x0 + sigma * eps
    return x_noisy, eps

plt.figure(figsize=(4, 2.5)); plt.plot(sigmas); plt.yscale("log"); plt.xlabel("level"); plt.ylabel("sigma"); plt.title("Noise levels"); plt.show()

**Check TODO 1.** With $\sigma = 0$ nothing should change. The plots show the spiral fading as $\sigma$ grows.

In [ ]:
x_same, _ = add_noise(data, 0.0)
assert torch.allclose(x_same, data), "with sigma = 0, add_noise must return the points unchanged"
x_n, e_n = add_noise(data, 0.5)
assert torch.allclose(x_n, data + 0.5 * e_n), "the noisy points must be x0 + sigma * eps, with the same eps you return"
print("TODO 1 works.")

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, s in zip(axes, [0.0, 0.05, 0.2, 1.0]):
    plot_points(add_noise(data, s)[0], ax=ax); ax.set_title(f"sigma = {s}")
plt.tight_layout(); plt.show()

---
## 3. The model: predict the noise

The network sees a noisy point and its noise level, and predicts the noise $\varepsilon$ that was added: $\varepsilon_\theta(x_\sigma, \sigma)$.

The noise level spans 10 to 0.005, which is a bad range for a network input. So we encode it as two bounded numbers, $[\sin(\tfrac{1}{2}\log\sigma),\ \cos(\tfrac{1}{2}\log\sigma)]$, and feed those next to the point. The network is a small MLP.

In [ ]:
def embed_sigma(sigma):
    s = torch.as_tensor(sigma, dtype=torch.float32).reshape(-1, 1)
    return torch.cat([torch.sin(torch.log(s) / 2), torch.cos(torch.log(s) / 2)], dim=1)

class NoisePredictor(nn.Module):
    def __init__(self, hidden=(16, 128, 128, 128, 128, 16)):
        super().__init__()
        dims, layers = (4,) + hidden, []
        for a, b in zip(dims[:-1], dims[1:]):
            layers += [nn.Linear(a, b), nn.GELU()]
        self.net = nn.Sequential(*layers, nn.Linear(dims[-1], 2))

    def forward(self, x, sigma):
        e = embed_sigma(sigma)
        if e.shape[0] == 1:
            e = e.expand(x.shape[0], 2)
        return self.net(torch.cat([x, e], dim=1))

model = NoisePredictor()
print(model)

---
## 4. Training

For each training step: pick a random noise level for every point, add that much noise, and ask the model for the noise. The loss is the squared error between the predicted noise and the true noise:

$$\mathcal{L}(\theta) = \mathbb{E}_{x_0,\ \sigma,\ \varepsilon}\Big[\big\lVert \varepsilon_\theta(x_0 + \sigma\varepsilon,\ \sigma) - \varepsilon \big\rVert^2\Big]$$

In [ ]:
def diffusion_loss(model, x0):
    # one random noise level per point, picked from the 200 levels
    sigma = sigmas[torch.randint(0, N_LEVELS, (x0.shape[0],))].unsqueeze(1)   # shape (n, 1)
    # TODO: add noise to x0 at these levels (use add_noise), predict the noise with model(x_noisy, sigma), and return the mean squared error against the true noise
    x_noisy, eps = add_noise(x0, sigma)
    eps_hat = model(x_noisy, sigma)
    return F.mse_loss(eps_hat, eps)

def train(model, steps=15000, lr=1e-3):
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    losses = []
    for _ in range(steps):
        loss = diffusion_loss(model, data)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    return losses

torch.manual_seed(0)
model = NoisePredictor()
t0 = time.time()
losses = train(model)
print(f"trained in {time.time() - t0:.0f} s")

plt.figure(figsize=(6, 3))
plt.plot(np.convolve(losses, np.ones(200) / 200, mode="valid"))
plt.yscale("log"); plt.xlabel("step"); plt.ylabel("loss (moving average)"); plt.show()
print("TODO 2 looks right." if np.mean(losses[-500:]) < 0.75 * np.mean(losses[:500]) else "The loss did not go down: check TODO 2.")

---
## 5. Sampling: from noise to the spiral

Start with pure noise at the top level, $x = \sigma_\text{max}\,\varepsilon$. Then walk down a list of decreasing noise levels $\sigma_{t} > \sigma_{t-1} > \dots > 0$. At each step, remove the noise the model predicts, scaled by how much the level drops:

$$x_{t-1} = x_t - (\sigma_t - \sigma_{t-1})\ \varepsilon_\theta(x_t, \sigma_t)$$

This is a deterministic sampler (DDIM). Once the starting noise is chosen, there is no more randomness.

In [ ]:
def sample_levels(steps):
    # `steps` levels from sigma_max down to sigma_min, spaced on a log scale, then a final 0
    lv = torch.exp(torch.linspace(math.log(SIGMA_MAX), math.log(SIGMA_MIN), steps))
    return torch.cat([lv, torch.zeros(1)])

@torch.no_grad()
def sample(model, steps=20, n=2000, x=None):
    levels = sample_levels(steps)
    x = levels[0] * torch.randn(n, 2) if x is None else x
    path = [x]
    for sig, sig_prev in zip(levels[:-1], levels[1:]):
        # TODO: one sampler step: x minus (sig - sig_prev) times the model's predicted noise at level sig
        x = x - (sig - sig_prev) * model(x, sig)
        path.append(x)
    return x, path

torch.manual_seed(1)
x_gen, path = sample(model, steps=20)
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
plot_points(data, ax=axes[0], c="lightgray", s=10); plot_points(x_gen, ax=axes[0], s=2, c="crimson")
axes[0].set_title("Gray: training points. Red: 2000 new samples")
# the paths of 30 samples, from noise to the spiral
P = torch.stack(path)[:, :30].numpy()
plot_points(data, ax=axes[1], c="lightgray", s=10)
for i in range(P.shape[1]):
    axes[1].plot(P[:, i, 0], P[:, i, 1], lw=0.8)
axes[1].set_title("Paths of 30 samples (they start far outside)")
plt.tight_layout(); plt.show()
near = torch.cdist(x_gen, data).min(dim=1).values
print(f"{(near < 0.1).float().mean():.0%} of samples land within 0.1 of a training point.",
      "TODO 3 looks right." if (near < 0.1).float().mean() > 0.6 else "Few samples near the spiral: check TODO 3.")

---
## 6. What did the model learn?

For a fixed noise level, draw the predicted denoising direction, $-\varepsilon_\theta(x, \sigma)$, at every point of a grid. At a large $\sigma$ the arrows point roughly toward the middle of the data. At a small $\sigma$ they point to the nearest part of the spiral.

In [ ]:
@torch.no_grad()
def plot_field(model, sigma, ax):
    g = np.linspace(-1.3, 1.3, 21)
    X = torch.tensor(np.array(np.meshgrid(g, g)).reshape(2, -1).T, dtype=torch.float32)
    d = -model(X, sigma)
    d = d / d.norm(dim=1, keepdim=True)
    plot_points(data, ax=ax, c="lightgray", s=8)
    ax.quiver(X[:, 0], X[:, 1], d[:, 0], d[:, 1], scale=30, width=0.004)
    ax.set_title(f"sigma = {sigma}")

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, s in zip(axes, [2.0, 0.3, 0.02]):
    plot_field(model, s, ax)
plt.tight_layout(); plt.show()

### Try it

1. **Steps.** Sample with `steps=5`, `20` and `100`. How do the samples change? What does fewer steps cost you, and what does it save?
2. **The arrows.** In Section 6, try other values of $\sigma$. Describe how the direction of the arrows changes as $\sigma$ goes from large to small.
3. **Memorize or generalize?** Train on far fewer points: `data = spiral(n=10)`, then re-run from Section 4. Do the new samples fill the spiral, or pile up on the 10 training points?
4. **Perception link.** Your visual system also starts from a noisy signal and settles on an interpretation. Where is that like this sampler, and where is it not?

---
## 7. If you have time: a faster sampler

The sampler above uses only the current prediction. A faster one mixes in the previous prediction, with a weight $\gamma \ge 1$:

$$\bar\varepsilon_t = \gamma\,\varepsilon_\theta(x_t, \sigma_t) + (1 - \gamma)\,\varepsilon_\theta(x_{t+1}, \sigma_{t+1}), \qquad x_{t-1} = x_t - (\sigma_t - \sigma_{t-1})\,\bar\varepsilon_t$$

On the first step there is no previous prediction, so use the current one. With $\gamma = 1$ this is the sampler from Section 5. Try $\gamma = 2$ with only 10 steps.

In [ ]:
@torch.no_grad()
def sample_fast(model, steps=10, n=2000, gamma=2.0):
    levels = sample_levels(steps)
    x = levels[0] * torch.randn(n, 2)
    eps_prev = None
    for sig, sig_prev in zip(levels[:-1], levels[1:]):
        eps = model(x, sig)
        # TODO: eps_bar = gamma * eps + (1 - gamma) * eps_prev, or just eps on the first step; then the update for x; then remember eps as eps_prev
        eps_bar = eps if eps_prev is None else gamma * eps + (1 - gamma) * eps_prev
        x = x - (sig - sig_prev) * eps_bar
        eps_prev = eps
    return x

torch.manual_seed(1)
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, g in zip(axes, [1.0, 2.0, 3.0]):
    plot_points(data, ax=ax, c="lightgray", s=10); plot_points(sample_fast(model, steps=10, gamma=g), ax=ax, s=2, c="crimson")
    ax.set_title(f"10 steps, gamma = {g}")
plt.tight_layout(); plt.show()